# 11 - Retrieval Telemetry Harness: Arm A0 Baseline (recall@k / MRR)

First real recall@k / MRR baseline on this revived account's populated S3 Vectors
index (614,647 vectors), using the actual production path
(`init_rag_components()` -> `run_supply_line_2_rag()`), not a synthetic probe.

**Arm A0 = current system, no reranking.** `reranked_*` metrics will be `None`
throughout -- there is no reranker yet (Item B, separate work). This is
retrieval-only: no Bedrock LLM synthesis call, no ~$0.017/query synthesis cost,
per `guidance/retrieval_telemetry_and_reranking_design.md`'s explicit recommendation
to call the supply line directly rather than `answer_query_batch()` for retrieval
measurement.

**Variants forced OFF in-memory only** (not touching `ml_config.yaml`) for this
baseline run -- `enable_variants: true` puts a nondeterministic Haiku rephrase call
upstream of retrieval, which would make this number irreproducible. A second pass
with variants on can confirm the effect survives, if this baseline result is
interesting enough to warrant it.

**Gold set:** all 31 questions in `p3_gold_test_suite_31q.json`, stratified by both
`retrieval_scope` and `gold_version` in the aggregate report -- `RETRIEVAL_IMPROVEMENT_STUDY.md`
Sec 7.4 documents real circularity concerns in 5 of the 31 `P3.v2` questions, so the
stratified breakdown is reported rather than curating those 5 out silently.

In [1]:
import sys
import json
import time
from pathlib import Path

for p in [Path.cwd()] + list(Path.cwd().parents):
    if p.name == "ModelPipeline":
        MODEL_ROOT = p
        break
if str(MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT))

from finrag_ml_tg1.rag_modules_src.synthesis_pipeline.supply_lines import (
    init_rag_components, run_supply_line_2_rag,
)
from finrag_ml_tg1.rag_modules_src.utilities.retrieval_metrics import score_query, aggregate

rag = init_rag_components()

# In-memory only override -- does not touch ml_config.yaml. See markdown above.
print(f"enable_variants before override: {rag.retriever.enable_variants}")
rag.retriever.enable_variants = False
print(f"enable_variants after override:  {rag.retriever.enable_variants}")

GOLD_PATH = MODEL_ROOT.parent / "MLFlow_POC" / "data" / "p3_gold_test_suite_31q.json"
gold = json.loads(GOLD_PATH.read_text())
print(f"\nLoaded {len(gold)} gold questions")
print(f"gold_version counts: "
      f"{sum(1 for q in gold if q['gold_version']=='P3.v2')} P3.v2, "
      f"{sum(1 for q in gold if q['gold_version']=='P3.v3')} P3.v3")

[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Local dev detected → LOCAL_CACHE mode
enable_variants before override: True
enable_variants after override:  False

Loaded 31 gold questions
gold_version counts: 21 P3.v2, 10 P3.v3


In [2]:
rows = []
t_start = time.time()

for i, gq in enumerate(gold, start=1):
    try:
        *_, telemetry = run_supply_line_2_rag(gq["question_text"], rag)
        row = score_query(telemetry, gq)
        row["gold_version"] = gq.get("gold_version")
        rows.append(row)
        print(f"[{i:2d}/{len(gold)}] {gq['question_id']:12s} "
              f"core_mrr={row['core_mrr']:.3f}  expanded_mrr={row['expanded_mrr']:.3f}  "
              f"union_hits={telemetry['counts']['union_hits']}")
    except Exception as e:
        print(f"[{i:2d}/{len(gold)}] {gq['question_id']:12s} FAILED: {type(e).__name__}: {e}")

elapsed = time.time() - t_start
print(f"\n{len(rows)}/{len(gold)} questions scored in {elapsed:.1f}s "
      f"({elapsed/max(1,len(rows)):.2f}s/query, retrieval-only, no synthesis cost)")

[ 1/31] P3V2-Q001    core_mrr=0.000  expanded_mrr=0.000  union_hits=30


[ 2/31] P3V2-Q002    core_mrr=0.045  expanded_mrr=0.014  union_hits=30


    ✗ CRITICAL: Core hit not found in window!
      Hit: 0000104169_10-K_2022_section_8_0 (pos=0)
      Window: [1, 3]
      This indicates data integrity issue.


    ✗ CRITICAL: Core hit not found in window!
      Hit: 0000104169_10-K_2024_section_8_0 (pos=0)
      Window: [1, 3]
      This indicates data integrity issue.


    ✗ CRITICAL: Core hit not found in window!
      Hit: 0000104169_10-K_2023_section_8_0 (pos=0)
      Window: [1, 3]
      This indicates data integrity issue.


...

    ✗ CRITICAL: Core hit not found in window!
      Hit: 0001318605_10-K_2022_section_8_0 (pos=0)
      Window: [1, 3]
      This indicates data integrity issue.


[28/31] P3V3-Q007    core_mrr=0.071  expanded_mrr=0.014  union_hits=30


[29/31] P3V3-Q008    core_mrr=0.000  expanded_mrr=0.000  union_hits=30


[30/31] P3V3-Q009    core_mrr=1.000  expanded_mrr=0.250  union_hits=30


[31/31] P3V3-Q010    core_mrr=0.111  expanded_mrr=0.018  union_hits=30

31/31 questions scored in 943.7s (30.44s/query, retrieval-only, no synthesis cost)


In [3]:
import polars as pl

df = pl.DataFrame(rows)
print(df.select([
    "question_id", "retrieval_scope", "gold_version", "n_evidence",
    "core_recall@5", "core_recall@30", "core_mrr",
    "expanded_recall@5", "expanded_recall@30", "expanded_mrr",
]))

print("\n=== Overall (n=%d) ===" % len(rows))
overall = aggregate(rows)
for k, v in overall["overall"].items():
    if isinstance(v, float):
        print(f"  {k:24s} {v:.4f}")
    else:
        print(f"  {k:24s} {v}")

print("\n=== By retrieval_scope ===")
by_scope = aggregate(rows, group_by="retrieval_scope")
for scope, block in by_scope["by_retrieval_scope"].items():
    print(f"  {scope} (n={block['n']}): core_recall@30={block.get('core_recall@30'):.3f}  "
          f"core_mrr={block.get('core_mrr'):.3f}  expanded_mrr={block.get('expanded_mrr'):.3f}")

print("\n=== By gold_version (circularity caveat -- see RETRIEVAL_IMPROVEMENT_STUDY.md Sec 7.4) ===")
by_version = aggregate(rows, group_by="gold_version")
for ver, block in by_version["by_gold_version"].items():
    print(f"  {ver} (n={block['n']}): core_recall@30={block.get('core_recall@30'):.3f}  "
          f"core_mrr={block.get('core_mrr'):.3f}  expanded_mrr={block.get('expanded_mrr'):.3f}")

shape: (31, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬──────────┬───────────┬───────────┬───────────┐
│ question_ ┆ retrieval ┆ gold_vers ┆ n_evidenc ┆ … ┆ core_mrr ┆ expanded_ ┆ expanded_ ┆ expanded_ │
│ id        ┆ _scope    ┆ ion       ┆ e         ┆   ┆ ---      ┆ recall@5  ┆ recall@30 ┆ mrr       │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ f64      ┆ ---       ┆ ---       ┆ ---       │
│ str       ┆ str       ┆ str       ┆ i64       ┆   ┆          ┆ f64       ┆ f64       ┆ f64       │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪══════════╪═══════════╪═══════════╪═══════════╡
│ P3V2-Q001 ┆ local     ┆ P3.v2     ┆ 1         ┆ … ┆ 0.0      ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ P3V2-Q002 ┆ local     ┆ P3.v2     ┆ 1         ┆ … ┆ 0.045455 ┆ 0.0       ┆ 0.0       ┆ 0.014493  │
│ P3V2-Q003 ┆ local     ┆ P3.v2     ┆ 1         ┆ … ┆ 0.0      ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ P3V2-Q004 ┆ local     ┆ P3.v2     ┆ 1         ┆ … ┆ 0.0      ┆ 0.0       

## Interpretation (fill in after running)

- This is **arm A0** -- the reference point every future arm (A1 score-only,
  A2 pruned) gets compared against via `paired_delta()`.
- `core` vs `expanded` gap is the measured value of the ±3 sentence window
  expansion -- a gold sentence the ANN retriever missed but the window rescued
  shows up as a core-stage miss but an expanded-stage hit.
- Per `guidance/ANALYSIS_reranker_judgment_calls_2026-07-29.md` Sec 2.8: with n=31,
  the minimum detectable MRR effect at 80% power is roughly +0.07 to +0.13 --
  read any future arm-vs-arm delta against that floor, not against zero.
- `retrieval_scope` cells of n=3/n=4 (cross_company/cross_year) are descriptive
  only, not statistically meaningful alone -- report them, do not test them.